> **Versión corregida (v2.13, línea v1 de despliegue, ejecución verificada)** — generada a partir de `notebooks/08_entrenamiento_ppo_autocontenido.ipynb` sin sobrescribirlo.
>
> Cambios aplicados:
> - Celda de inicialización: se sustituyen la detección/montaje de Drive y el clonado de `main` por el bootstrap portable (`PROJECT_ROOT`/`REPO_ROOT`, rama `feature/entorno-capacidad-v2`).
> - Renombrado semántico: el antiguo `PROJECT_ROOT` (copia del repositorio) pasa a `REPO_ROOT`; `PERSISTENT_ROOT` se conserva como raíz de la línea v1 (= `PROJECT_ROOT`, o `PROJECT_ROOT/prueba_linea_v1` en modo prueba).
> - Modo prueba: submuestra estratificada por `categoria_riesgo` (sin crear columnas); la capacidad es relativa (0.20), por lo que no requiere escalarse. La cohorte siempre se lee de `PROJECT_ROOT/data/processed/cohorte_diabetes.csv`.
> - Modo prueba: 4 096 pasos totales (completo: 100 000).
>
> Rutas: `PROJECT_ROOT` = datos/modelos/resultados (Drive en Colab, o `PIDA_PROJECT_ROOT`); `REPO_ROOT` = código. Modo prueba: `PIDA_MODO_PRUEBA=1` (cohorte pequeña, pocos pasos, salida en `prueba_linea_v1/`).


# PIDA-RL Diabetes - 08. Entrenamiento y evaluacion de PPO

Este notebook entrena y evalua Proximal Policy Optimization (PPO) sobre el mismo simulador utilizado para las politicas base, DQN y Q-learning. Es autocontenido y utiliza solo la cohorte persistente en Google Drive.

> Los resultados son experimentos dentro de un simulador. No son evidencia clinica ni recomendaciones individuales de atencion.


## 1. Inicializacion


In [ ]:
# === Bootstrap portable: PROJECT_ROOT (corrección v2.12) ===
# PROJECT_ROOT = raíz persistente de datos, modelos y resultados.
# REPO_ROOT    = copia local del código (repositorio GitHub).
# Prioridad: variables de entorno PIDA_PROJECT_ROOT / PIDA_REPO_ROOT,
# después Google Colab + Drive y, por último, ejecución local.
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import importlib
import importlib.util
import subprocess

PROJECT_NAME = 'pida-rl-datos-publicos'

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'feature/entorno-capacidad-v2')
PROJECT_FOLDER_NAME = 'PIDA-RL-Diabetes'
IN_COLAB = 'google.colab' in sys.modules

DRIVE_ROOT = None
if os.environ.get('PIDA_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['PIDA_PROJECT_ROOT']).expanduser().resolve()
elif IN_COLAB:
    DRIVE_ROOT = next(
        (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
        None,
    )
    if DRIVE_ROOT is None:
        from google.colab import drive
        mount_point = Path('/content/drive')
        if mount_point.exists() and any(mount_point.iterdir()):
            mount_point = Path('/content/gdrive')
        drive.mount(str(mount_point), force_remount=False)
        DRIVE_ROOT = mount_point
    PROJECT_ROOT = DRIVE_ROOT / 'MyDrive' / PROJECT_FOLDER_NAME
else:
    PROJECT_ROOT = (Path.cwd() / PROJECT_FOLDER_NAME).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


def _buscar_repo(inicio):
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'src' / 'environment' / 'diabetes_followup_env.py').is_file():
            return candidato
    return None


if os.environ.get('PIDA_REPO_ROOT'):
    REPO_ROOT = Path(os.environ['PIDA_REPO_ROOT']).expanduser().resolve()
else:
    REPO_ROOT = _buscar_repo(Path.cwd().resolve()) or (
        Path('/content/pida-rl-datos-publicos') if IN_COLAB
        else Path.cwd().resolve() / 'pida-rl-datos-publicos'
    )
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])
elif not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(f'REPO_ROOT no contiene la carpeta src: {REPO_ROOT}')
elif IN_COLAB and (REPO_ROOT / '.git').is_dir():
    # En Colab se alinea la copia temporal con la rama de trabajo (sin push ni commit).
    for args in (['fetch', 'origin', BRANCH], ['checkout', BRANCH], ['pull', '--ff-only', 'origin', BRANCH]):
        subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=False)

SRC_ROOT = REPO_ROOT / 'src'
for ruta in (str(REPO_ROOT), str(SRC_ROOT)):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)
os.chdir(REPO_ROOT)

# Estructura persistente común (antes: rutas fijas /content/drive/MyDrive/PIDA-RL-Diabetes).
PERSISTENT_ROOT = PROJECT_ROOT  # alias de compatibilidad con los notebooks originales
MYDRIVE_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
SPECIFICATIONS_DIR = DATA_ROOT / 'specifications'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

# Modo prueba: cohorte pequeña y pocos episodios (PIDA_MODO_PRUEBA=1).
MODO_PRUEBA = os.environ.get('PIDA_MODO_PRUEBA', '0') == '1'
N_COHORTE_PRUEBA = int(os.environ.get('PIDA_N_COHORTE', '120'))
SEMILLA_MUESTRA_PRUEBA = 2026
RESULTS_V2_DIR = RESULTS_DIR / ('capacidad_v2_prueba' if MODO_PRUEBA else 'capacidad_v2')


def muestra_cohorte_prueba(cohorte, n=None, semilla=SEMILLA_MUESTRA_PRUEBA):
    """Devuelve (cohorte, factor). En modo prueba toma una submuestra
    estratificada por categoria_riesgo (bajo/medio/alto); no crea columnas."""
    n = N_COHORTE_PRUEBA if n is None else int(n)
    if not MODO_PRUEBA or len(cohorte) <= n:
        return cohorte, 1.0
    fraccion = n / len(cohorte)
    muestra = (
        cohorte.groupby('categoria_riesgo', group_keys=False)
        .sample(frac=fraccion, random_state=semilla)
        .sort_values('FOLIO_INT')
        .reset_index(drop=True)
    )
    return muestra, len(muestra) / len(cohorte)


def escalar_capacidad(capacidad, factor):
    """Mantiene la presión de capacidad (recursos/persona) al reducir la cohorte."""
    return capacidad if factor >= 1.0 else max(1, int(round(capacidad * factor)))

print('Entorno:', 'Google Colab' if IN_COLAB else 'local')
print('PROJECT_ROOT (datos/resultados):', PROJECT_ROOT)
print('REPO_ROOT (código):', REPO_ROOT)
print('Modo prueba:', MODO_PRUEBA)

# --- Línea v1 de despliegue (entorno autocontenido) ---
# En modo prueba se escribe en una carpeta aparte para no sobrescribir resultados completos ni el dashboard publicado.
LINEA_V1_ROOT = PROJECT_ROOT / 'prueba_linea_v1' if MODO_PRUEBA else PROJECT_ROOT
PERSISTENT_ROOT = LINEA_V1_ROOT
COHORTE_CANONICA = PROJECT_ROOT / 'data' / 'processed' / 'cohorte_diabetes.csv'
print('Raíz de la línea v1 (PERSISTENT_ROOT):', PERSISTENT_ROOT)

# --- Celda original (sin montaje de Drive ni clonado fijo) ---
from pathlib import Path
import os
import sys
import subprocess
import json

DATA_ROOT = PERSISTENT_ROOT / 'data'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
MODELS_DIR = PERSISTENT_ROOT / 'models'
TABLES_DIR = PERSISTENT_ROOT / 'results' / 'tables'
LOGS_DIR = PERSISTENT_ROOT / 'logs'

for directory in [MODELS_DIR, TABLES_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_ROOT)

COHORT_PATH = COHORTE_CANONICA
BASELINES_PATH = TABLES_DIR / 'evaluacion_politicas_base_v1.csv'
DQN_PATH = TABLES_DIR / 'evaluacion_dqn_v1.csv'
Q_LEARNING_PATH = TABLES_DIR / 'evaluacion_q_learning_v1.csv'

if not COHORT_PATH.is_file():
    raise FileNotFoundError(f'No existe la cohorte: {COHORT_PATH}. Ejecuta primero el notebook 03.')

print('Proyecto temporal:', REPO_ROOT)
print('Cohorte persistente:', COHORT_PATH)
print('Base disponible:', BASELINES_PATH.is_file())
print('DQN disponible:', DQN_PATH.is_file())
print('Q-learning disponible:', Q_LEARNING_PATH.is_file())


## 2. Cargar y validar la cohorte

Esta validacion protege el entrenamiento contra valores faltantes o infinitos que pueden provocar inestabilidad numerica en PPO.


In [ ]:
import numpy as np
import pandas as pd

df_cohorte_diabetes = pd.read_csv(COHORT_PATH, dtype={'FOLIO_INT': 'string'})
df_cohorte_diabetes, FACTOR_COHORTE = muestra_cohorte_prueba(df_cohorte_diabetes)
if MODO_PRUEBA:
    print('Modo prueba: cohorte estratificada de', len(df_cohorte_diabetes), 'perfiles (capacidad relativa 0.20 sin cambios).')

required_columns = [
    'FOLIO_INT', 'edad', 'sexo', 'anios_con_diabetes',
    'insulina_diaria', 'depresion', 'consultas_control_12m',
    'num_hospitalizaciones', 'num_complicaciones',
    'score_riesgo', 'categoria_riesgo'
]

missing = [column for column in required_columns if column not in df_cohorte_diabetes.columns]
if missing:
    raise KeyError('Faltan columnas en cohorte: ' + ', '.join(missing))

numeric_columns = [
    'edad', 'sexo', 'anios_con_diabetes', 'insulina_diaria',
    'depresion', 'consultas_control_12m', 'num_hospitalizaciones',
    'num_complicaciones', 'score_riesgo'
]

values = df_cohorte_diabetes[numeric_columns].to_numpy(dtype=float)
assert np.isfinite(values).all(), 'La cohorte contiene NaN o infinitos.'
assert df_cohorte_diabetes['categoria_riesgo'].isin(['bajo', 'medio', 'alto']).all()
assert not df_cohorte_diabetes['FOLIO_INT'].duplicated().any()

print('Cohorte valida:', df_cohorte_diabetes.shape)
display(df_cohorte_diabetes[numeric_columns].isna().sum().rename('faltantes'))
display(df_cohorte_diabetes['categoria_riesgo'].value_counts().rename_axis('riesgo').reset_index(name='n'))


## 3. Entorno autocontenido

La definicion replica las reglas empleadas en los notebooks 05, 06 y 07: estado de 13 elementos, cuatro acciones, presupuesto mensual de 20 por ciento y horizonte de 12 meses.


In [ ]:
try:
    import gymnasium as gym
    from gymnasium import spaces
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'gymnasium'])
    import gymnasium as gym
    from gymnasium import spaces

class DiabetesFollowUpEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self, cohorte_df, capacidad_mensual=0.20, horizonte=12, seed=None):
        super().__init__()
        required = [
            'edad', 'sexo', 'anios_con_diabetes', 'insulina_diaria',
            'depresion', 'consultas_control_12m', 'num_hospitalizaciones',
            'num_complicaciones', 'score_riesgo', 'categoria_riesgo'
        ]
        missing = [name for name in required if name not in cohorte_df.columns]
        if missing:
            raise KeyError('Faltan columnas: ' + ', '.join(missing))
        if not 0 < float(capacidad_mensual) <= 1:
            raise ValueError('capacidad_mensual debe estar entre 0 y 1.')
        if int(horizonte) < 1:
            raise ValueError('horizonte debe ser al menos 1.')
        self.cohorte_base = cohorte_df.reset_index(drop=True).copy()
        self.n_perfiles = len(self.cohorte_base)
        if self.n_perfiles == 0:
            raise ValueError('La cohorte no puede estar vacia.')
        numeric = [
            'edad', 'sexo', 'anios_con_diabetes', 'insulina_diaria',
            'depresion', 'consultas_control_12m', 'num_hospitalizaciones',
            'num_complicaciones', 'score_riesgo'
        ]
        if not np.isfinite(self.cohorte_base[numeric].to_numpy(dtype=float)).all():
            raise ValueError('La cohorte contiene NaN o infinitos.')
        if not self.cohorte_base['categoria_riesgo'].isin({'bajo', 'medio', 'alto'}).all():
            raise ValueError('categoria_riesgo invalida.')
        self.capacidad_mensual = float(capacidad_mensual)
        self.horizonte = int(horizonte)
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(13,), dtype=np.float32)
        self.costo_accion = {0: 0, 1: 1, 2: 2, 3: 3}
        self.reduccion_evento = {0: 0.00, 1: 0.20, 2: 0.45, 3: 0.65}
        self.riesgo_ordinal = {'bajo': 0, 'medio': 1, 'alto': 2}
        self.reset(seed=seed)

    @staticmethod
    def _norm(value, denominator):
        return float(np.clip(float(value) / max(float(denominator), 1.0), 0.0, 1.0))

    def _risk_level(self, category):
        return self.riesgo_ordinal[str(category)]

    def _get_obs(self, idx):
        row = self.cohorte.iloc[idx]
        remaining = self.presupuesto_mensual - self.recursos_usados_mes
        sex_norm = 0.0 if float(row['sexo']) == 1.0 else 1.0
        risk_norm = self._risk_level(row['categoria_riesgo']) / 2.0
        return np.array([
            self._norm(row['edad'], 100),
            sex_norm,
            self._norm(row['anios_con_diabetes'], 80),
            float(np.clip(row['insulina_diaria'], 0, 1)),
            float(np.clip(row['depresion'], 0, 1)),
            self._norm(row['consultas_control_12m'], 24),
            self._norm(row['num_hospitalizaciones'], 12),
            self._norm(row['num_complicaciones'], 5),
            self._norm(row['score_riesgo'], 16),
            risk_norm,
            self._norm(row['meses_sin_contacto'], self.horizonte),
            self._norm(self.mes_actual, self.horizonte),
            self._norm(remaining, self.presupuesto_mensual)
        ], dtype=np.float32)

    def _event_probability(self, row, risk_level):
        value = (
            0.01 + 0.025 * risk_level
            + 0.010 * float(row['num_complicaciones'])
            + 0.020 * float(row['num_hospitalizaciones'] > 0)
            + 0.005 * min(float(row['meses_sin_contacto']), 6)
        )
        return float(np.clip(value, 0.0, 0.80))

    def _reward(self, action, risk_level, action_accepted, adverse_event, over_capacity):
        reward = 0.0
        if action_accepted and action > 0:
            reward += {1: 0.5, 2: 1.0, 3: 1.5}[action] * (risk_level + 1)
        if action == 0 or not action_accepted:
            reward -= {0: 0.0, 1: 0.5, 2: 1.5}[risk_level]
        if adverse_event:
            reward -= 5.0
        if over_capacity:
            reward -= 3.0
        return float(reward)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.cohorte = self.cohorte_base.copy()
        self.cohorte['meses_sin_contacto'] = 0
        self.mes_actual = 1
        self.idx_actual = 0
        self.presupuesto_mensual = max(1, int(round(self.n_perfiles * self.capacidad_mensual)))
        self.recursos_usados_mes = 0
        self.contactos_aceptados_mes = 0
        self.intentos_exceso_mes = 0
        obs = self._get_obs(self.idx_actual)
        info = {
            'mes': self.mes_actual,
            'indice_perfil': self.idx_actual,
            'n_perfiles': self.n_perfiles,
            'presupuesto_mensual': self.presupuesto_mensual,
            'recursos_usados_mes': 0,
            'recursos_restantes_mes': self.presupuesto_mensual
        }
        return obs, info

    def step(self, action):
        action = int(action)
        if not self.action_space.contains(action):
            raise ValueError(f'Accion invalida: {action}')
        idx = self.idx_actual
        month = self.mes_actual
        row = self.cohorte.iloc[idx]
        risk_level = self._risk_level(row['categoria_riesgo'])
        requested_cost = self.costo_accion[action]
        over_capacity = self.recursos_usados_mes + requested_cost > self.presupuesto_mensual
        action_accepted = not over_capacity
        actual_cost = requested_cost if action_accepted else 0
        if over_capacity:
            self.intentos_exceso_mes += 1
        else:
            self.recursos_usados_mes += actual_cost
            if action > 0:
                self.contactos_aceptados_mes += 1
        base_probability = self._event_probability(row, risk_level)
        reduction = self.reduccion_evento[action] if action_accepted else 0.0
        probability = float(np.clip(base_probability * (1.0 - reduction), 0.0, 1.0))
        adverse_event = bool(self.np_random.random() < probability)
        reward = self._reward(action, risk_level, action_accepted, adverse_event, over_capacity)
        if action > 0 and action_accepted:
            self.cohorte.loc[idx, 'meses_sin_contacto'] = 0
        else:
            self.cohorte.loc[idx, 'meses_sin_contacto'] += 1
        used = int(self.recursos_usados_mes)
        remaining = int(self.presupuesto_mensual - self.recursos_usados_mes)
        contacts = int(self.contactos_aceptados_mes)
        excess_attempts = int(self.intentos_exceso_mes)
        self.idx_actual += 1
        terminated = False
        truncated = False
        month_changed = False
        if self.idx_actual >= self.n_perfiles:
            self.idx_actual = 0
            self.mes_actual += 1
            month_changed = True
            self.recursos_usados_mes = 0
            self.contactos_aceptados_mes = 0
            self.intentos_exceso_mes = 0
            if self.mes_actual > self.horizonte:
                terminated = True
        obs = np.zeros(self.observation_space.shape, dtype=np.float32) if terminated else self._get_obs(self.idx_actual)
        info = {
            'mes': month,
            'mes_siguiente': self.mes_actual,
            'cambio_mes': month_changed,
            'indice_perfil': idx,
            'nivel_riesgo': risk_level,
            'categoria_riesgo': str(row['categoria_riesgo']),
            'evento_adverso': adverse_event,
            'prob_evento': probability,
            'excede_capacidad': over_capacity,
            'accion_aceptada': action_accepted,
            'costo_solicitado': requested_cost,
            'costo_real': actual_cost,
            'presupuesto_mensual': self.presupuesto_mensual,
            'recursos_usados_mes': used,
            'recursos_restantes_mes': remaining,
            'contactos_aceptados_mes': contacts,
            'intentos_exceso_mes': excess_attempts
        }
        return obs, reward, terminated, truncated, info

print('Entorno PPO definido.')


## 4. Validar entorno antes de PPO

La prueba recorre pasos aleatorios y usa `VecCheckNan` para detectar observaciones, recompensas o acciones no finitas antes de iniciar el entrenamiento.


In [ ]:
try:
    import stable_baselines3
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'stable-baselines3'])

from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecCheckNan

env_check = DiabetesFollowUpEnv(df_cohorte_diabetes, capacidad_mensual=0.20, horizonte=12, seed=42)
check_env(env_check, warn=True)

obs, info = env_check.reset(seed=42)
for step in range(500):
    action = env_check.action_space.sample()
    obs, reward, terminated, truncated, info = env_check.step(action)
    assert np.isfinite(obs).all(), f'Observacion no finita en paso {step}'
    assert np.isfinite(reward), f'Recompensa no finita en paso {step}'
    if terminated or truncated:
        break

print('Entorno validado para PPO.')
print('Espacio:', env_check.observation_space, env_check.action_space)


## 5. Crear PPO y prueba corta

Se ejecuta primero una prueba de 2048 pasos. Solo si finaliza sin valores invalidos se realiza la corrida completa.


In [ ]:
from stable_baselines3 import PPO

def make_env(seed):
    def _init():
        return DiabetesFollowUpEnv(
            cohorte_df=df_cohorte_diabetes,
            capacidad_mensual=0.20,
            horizonte=12,
            seed=seed
        )
    return _init

env_ppo = DummyVecEnv([make_env(42)])
env_ppo = VecCheckNan(env_ppo, raise_exception=True)

PPO_MODEL_PATH = MODELS_DIR / 'ppo_diabetes_followup_v1'

model_ppo = PPO(
    policy='MlpPolicy',
    env=env_ppo,
    learning_rate=1e-4,
    n_steps=512,
    batch_size=64,
    n_epochs=5,
    gamma=0.95,
    gae_lambda=0.95,
    clip_range=0.20,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs={'net_arch': {'pi': [128, 128], 'vf': [128, 128]}},
    tensorboard_log=str(LOGS_DIR / 'ppo_v1'),
    verbose=1,
    seed=42,
    device='auto'
)

model_ppo.learn(total_timesteps=2048, progress_bar=True)
print('Prueba corta PPO completada.')


## 6. Entrenamiento completo

El entrenamiento total objetivo es 100000 pasos. La corrida continua desde la prueba corta sin reiniciar pesos ni contador.


In [ ]:
TOTAL_TIMESTEPS = int(os.environ.get('PIDA_PASOS_PPO', '4096')) if MODO_PRUEBA else 100000
SHORT_RUN_TIMESTEPS = 2048
REMAINING_TIMESTEPS = TOTAL_TIMESTEPS - SHORT_RUN_TIMESTEPS

if REMAINING_TIMESTEPS > 0:
    model_ppo.learn(
        total_timesteps=REMAINING_TIMESTEPS,
        reset_num_timesteps=False,
        progress_bar=True
    )

model_ppo.save(str(PPO_MODEL_PATH))

saved_model = Path(str(PPO_MODEL_PATH) + '.zip')
assert saved_model.is_file()
print('Modelo PPO guardado:', saved_model)
print('Bytes:', saved_model.stat().st_size)


## 7. Evaluar PPO

La politica se evalua de manera determinista con las mismas semillas utilizadas por las politicas base, DQN y Q-learning.


In [ ]:
def evaluate_ppo(model, cohort, seeds=(42, 123, 2026)):
    summary_rows = []
    step_rows = []

    for seed in seeds:
        env = DiabetesFollowUpEnv(cohort, capacidad_mensual=0.20, horizonte=12, seed=seed)
        obs, _ = env.reset(seed=seed)
        total_reward = 0.0
        episode_rows = []
        step_number = 0

        while True:
            action, _ = model.predict(obs, deterministic=True)
            action = int(action)
            next_obs, reward, terminated, truncated, info = env.step(action)
            step_number += 1
            total_reward += float(reward)

            row = {
                'politica': 'ppo_v1',
                'semilla': int(seed),
                'paso': int(step_number),
                'mes': int(info['mes']),
                'nivel_riesgo': int(info['nivel_riesgo']),
                'categoria_riesgo': info['categoria_riesgo'],
                'accion': action,
                'accion_aceptada': bool(info['accion_aceptada']),
                'contacto_aceptado': bool(action > 0 and info['accion_aceptada']),
                'evento_adverso': bool(info['evento_adverso']),
                'recompensa': float(reward),
                'costo_real': int(info['costo_real']),
                'excede_capacidad': bool(info['excede_capacidad'])
            }

            episode_rows.append(row)
            step_rows.append(row)
            obs = next_obs

            if terminated or truncated:
                break

        episode = pd.DataFrame(episode_rows)
        grouped = episode.groupby(['nivel_riesgo', 'categoria_riesgo'], as_index=False).agg(
            decisiones=('accion', 'size'),
            contactos_aceptados=('contacto_aceptado', 'sum'),
            eventos_adversos=('evento_adverso', 'sum'),
            recursos_usados=('costo_real', 'sum'),
            excesos_capacidad=('excede_capacidad', 'sum')
        )

        grouped['cobertura_porcentaje'] = np.where(
            grouped['decisiones'] > 0,
            100.0 * grouped['contactos_aceptados'] / grouped['decisiones'],
            0.0
        )

        grouped['politica'] = 'ppo_v1'
        grouped['semilla'] = int(seed)
        grouped['recompensa_acumulada_episodio'] = float(total_reward)
        grouped['pasos_episodio'] = int(step_number)
        summary_rows.extend(grouped.to_dict(orient='records'))

    return pd.DataFrame(summary_rows), pd.DataFrame(step_rows)

df_ppo_summary, df_ppo_steps = evaluate_ppo(model_ppo, df_cohorte_diabetes)
display(df_ppo_summary.sort_values(['semilla', 'nivel_riesgo']))


## 8. Diagnostico, comparacion y persistencia


In [ ]:
df_ppo_action_distribution = (
    df_ppo_steps
    .groupby(['categoria_riesgo', 'accion'], as_index=False)
    .size()
    .rename(columns={'size': 'frecuencia'})
)

df_ppo_action_distribution['porcentaje_en_riesgo'] = (
    100.0 * df_ppo_action_distribution['frecuencia']
    / df_ppo_action_distribution.groupby('categoria_riesgo')['frecuencia'].transform('sum')
)

display(df_ppo_action_distribution.sort_values(['categoria_riesgo', 'accion']))

PPO_SUMMARY_PATH = TABLES_DIR / 'evaluacion_ppo_v1.csv'
PPO_STEPS_PATH = TABLES_DIR / 'evaluacion_pasos_ppo_v1.csv'
PPO_DIAGNOSTIC_PATH = TABLES_DIR / 'diagnostico_acciones_ppo_v1.csv'

df_ppo_summary.to_csv(PPO_SUMMARY_PATH, index=False, encoding='utf-8-sig')
df_ppo_steps.to_csv(PPO_STEPS_PATH, index=False, encoding='utf-8-sig')
df_ppo_action_distribution.to_csv(PPO_DIAGNOSTIC_PATH, index=False, encoding='utf-8-sig')

comparison_inputs = []
for path in [BASELINES_PATH, DQN_PATH, Q_LEARNING_PATH, PPO_SUMMARY_PATH]:
    if path.is_file():
        comparison_inputs.append(pd.read_csv(path))

if comparison_inputs:
    df_comparison_available = pd.concat(comparison_inputs, ignore_index=True)
    comparison_path = TABLES_DIR / 'comparacion_modelos_disponibles_v1.csv'
    df_comparison_available.to_csv(comparison_path, index=False, encoding='utf-8-sig')
    print('Comparacion guardada:', comparison_path)

for path in [PPO_SUMMARY_PATH, PPO_STEPS_PATH, PPO_DIAGNOSTIC_PATH]:
    print(path.name, '| existe:', path.is_file(), '| bytes:', path.stat().st_size if path.is_file() else 0)

assert np.isfinite(df_ppo_steps['recompensa']).all()
assert (df_ppo_steps['costo_real'] >= 0).all()
print('PPO entrenado, evaluado y guardado.')


## Resultado esperado

Deben existir el modelo `ppo_diabetes_followup_v1.zip`, las tablas de evaluacion y el diagnostico de acciones. Si PPO presenta inestabilidad numerica, conserva el error y los resultados previos: sera una limitacion documentada.
